# 06 — Scenario Generation\n\nGenerate future scenarios based on CIB-consistent driver combinations.\n- **Evolutionary** scenarios from validated/incremental drivers\n- **Disruptive** scenarios from speculative drivers\n\nEach scenario references its driver assumptions + sources.

In [1]:
import sys, os, json
import numpy as np
sys.path.insert(0, os.path.abspath(".."))

from src.config import CHROMA_PERSIST_DIR
from src.llm import safe_chat_json, embed
from src.traceability import TechDriver, Scenario, DriverAssumption, ScenarioType, DriverOrigin
from src import prompts

import chromadb
client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
collection = client.get_collection("knowledge_base")

with open("../data/outputs/merge_state.json") as f:
    drivers = [TechDriver(**d) for d in json.load(f)["unified_drivers"]]

with open("../data/outputs/cib_state.json") as f:
    cib = json.load(f)

cib_matrix = np.array(cib["matrix"])
cib_driver_ids = cib["driver_ids"]
cib_id_to_idx = {did: i for i, did in enumerate(cib_driver_ids)}

print(f"Drivers: {len(drivers)}")
print(f"CIB matrix: {cib_matrix.shape}")
print(f"CIB influence ranking:")
for did, score in sorted(cib["influence"].items(), key=lambda x: x[1], reverse=True)[:5]:
    d = next((x for x in drivers if x.id == did), None)
    if d:
        print(f"  {score:+3d} | {d.name}")

Drivers: 27
CIB matrix: (15, 15)
CIB influence ranking:
  +14 | Internal GNSS Module
   +8 | Filtering Architecture
   +6 | Real-Time Bandwidth Capture and Photonic Signal Processing for Ultra-Wideband Receivers
   +6 | Dynamic Range Enhancement System and Energy-Efficient AI Architectures for Spectrum Monitoring
   +5 | I/Q Data Streaming Interface and Edge Computing for Distributed Spectrum Monitoring


In [2]:
# categorize drivers for diverse scenario archetypes
validated = [d for d in drivers if d.origin == DriverOrigin.BOTH]
incremental = [d for d in drivers if d.origin == DriverOrigin.BOM]
speculative = [d for d in drivers if d.origin == DriverOrigin.TREND]

ai_keywords = ["ai", "machine learning", "deep learning", "cognitive", "federated", "intelligent"]
space_keywords = ["satellite", "leo", "space"]
hardware_keywords = ["filter", "gnss", "tdoa", "aoa", "bandwidth", "downconversion"]

def matches(driver, keywords):
    name_desc = (driver.name + " " + driver.description).lower()
    return any(k in name_desc for k in keywords)

ai_drivers = [d for d in drivers if matches(d, ai_keywords)]
space_drivers = [d for d in drivers if matches(d, space_keywords)]
hw_drivers = [d for d in drivers if matches(d, hardware_keywords)]

print(f"Validated: {len(validated)}, Incremental: {len(incremental)}, Speculative: {len(speculative)}")
print(f"AI-related: {len(ai_drivers)}, Space-related: {len(space_drivers)}, Hardware-related: {len(hw_drivers)}")

# 6 scenario archetypes with mixed driver states
evo_drivers = (validated + incremental)[:6]
scenario_configs = []

# 1. Evolutionary Optimistic — validated+BOM, all breakthrough
scenario_configs.append({
    "name": "Evolutionary Optimistic",
    "type": ScenarioType.EVOLUTIONARY,
    "drivers": evo_drivers,
    "states": {d.id: "breakthrough" for d in evo_drivers},
})

# 2. Evolutionary Conservative — same drivers, all steady
scenario_configs.append({
    "name": "Evolutionary Conservative",
    "type": ScenarioType.EVOLUTIONARY,
    "drivers": evo_drivers,
    "states": {d.id: "steady_progress" for d in evo_drivers},
})

# 3. AI Transformation — AI drivers breakthrough, hardware stagnation
ai_set = ai_drivers[:4]
hw_set = [d for d in hw_drivers if d not in ai_set][:3]
ai_transform_drivers = ai_set + hw_set
scenario_configs.append({
    "name": "AI-Driven Transformation",
    "type": ScenarioType.DISRUPTIVE,
    "drivers": ai_transform_drivers,
    "states": {**{d.id: "breakthrough" for d in ai_set}, **{d.id: "stagnation" for d in hw_set}},
})

# 4. Hardware-Led + Regulatory Push — BOM hardware breakthrough, speculative steady
hw_push = (validated + incremental)[:4]
spec_steady = speculative[:3]
hw_reg_drivers = hw_push + spec_steady
scenario_configs.append({
    "name": "Hardware Evolution + Regulatory Push",
    "type": ScenarioType.EVOLUTIONARY,
    "drivers": hw_reg_drivers,
    "states": {**{d.id: "breakthrough" for d in hw_push}, **{d.id: "steady_progress" for d in spec_steady}},
})

# 5. Space+AI Disruption — space+AI breakthrough, ground hardware stagnation
space_ai = (space_drivers + [d for d in ai_drivers if d not in space_drivers])[:4]
ground_hw = [d for d in hw_drivers if d not in space_ai][:3]
space_disruption_drivers = space_ai + ground_hw
scenario_configs.append({
    "name": "Space + AI Disruption",
    "type": ScenarioType.DISRUPTIVE,
    "drivers": space_disruption_drivers,
    "states": {**{d.id: "breakthrough" for d in space_ai}, **{d.id: "stagnation" for d in ground_hw}},
})

# 6. Fragmented Progress — mixed: some breakthrough, some stagnation, some steady
import random
random.seed(42)
frag_drivers = (validated + incremental[:2] + speculative[:3])[:7]
frag_states = ["breakthrough", "stagnation", "steady_progress"]
scenario_configs.append({
    "name": "Fragmented Progress",
    "type": ScenarioType.DISRUPTIVE,
    "drivers": frag_drivers,
    "states": {d.id: frag_states[i % 3] for i, d in enumerate(frag_drivers)},
})

print(f"\nWill generate {len(scenario_configs)} scenarios:")
for sc in scenario_configs:
    state_counts = {}
    for s in sc["states"].values():
        state_counts[s] = state_counts.get(s, 0) + 1
    print(f"  [{sc['type'].value:14s}] {sc['name']} — {len(sc['drivers'])} drivers — {state_counts}")

Validated: 4, Incremental: 7, Speculative: 16
AI-related: 20, Space-related: 5, Hardware-related: 12

Will generate 6 scenarios:
  [evolutionary  ] Evolutionary Optimistic — 6 drivers — {'breakthrough': 6}
  [evolutionary  ] Evolutionary Conservative — 6 drivers — {'steady_progress': 6}
  [disruptive    ] AI-Driven Transformation — 7 drivers — {'breakthrough': 4, 'stagnation': 3}
  [evolutionary  ] Hardware Evolution + Regulatory Push — 7 drivers — {'breakthrough': 4, 'steady_progress': 3}
  [disruptive    ] Space + AI Disruption — 7 drivers — {'breakthrough': 4, 'stagnation': 3}
  [disruptive    ] Fragmented Progress — 7 drivers — {'breakthrough': 3, 'stagnation': 2, 'steady_progress': 2}


## CIB Consistency Check

Validate driver-state combinations against the CIB matrix.
If Driver A = "breakthrough" and CIB[A→B] ≤ -2, then B cannot also be "breakthrough" — downgrade to "steady_progress".

In [3]:
def check_cib_consistency(driver_ids: list[str], states: list[str], threshold: int = -2) -> list[str]:
    """Check and fix contradictions: if A='breakthrough' and CIB[A→B] <= threshold, B can't be 'breakthrough'."""
    adjusted = list(states)
    adjustments = []

    for i, (did_a, state_a) in enumerate(zip(driver_ids, adjusted)):
        if state_a != "breakthrough":
            continue
        idx_a = cib_id_to_idx.get(did_a)
        if idx_a is None:
            continue

        for j, (did_b, state_b) in enumerate(zip(driver_ids, adjusted)):
            if i == j or state_b != "breakthrough":
                continue
            idx_b = cib_id_to_idx.get(did_b)
            if idx_b is None:
                continue

            score = int(cib_matrix[idx_a][idx_b])
            if score <= threshold:
                adjusted[j] = "steady_progress"
                da = next((d for d in drivers if d.id == did_a), None)
                db = next((d for d in drivers if d.id == did_b), None)
                adjustments.append(
                    f"  CIB[{da.name[:30] if da else did_a} → {db.name[:30] if db else did_b}] = {score}: "
                    f"downgraded '{db.name[:30] if db else did_b}' to steady_progress"
                )

    return adjusted, adjustments


# apply consistency check to each config
print("=== CIB Consistency Check ===\n")
for config in scenario_configs:
    d_ids = [d.id for d in config["drivers"]]
    states = [config["states"][d.id] for d in config["drivers"]]

    adjusted_states, adj_log = check_cib_consistency(d_ids, states)

    if adj_log:
        print(f"[{config['name']}] — {len(adj_log)} adjustment(s):")
        for line in adj_log:
            print(line)
        config["adjusted_states"] = adjusted_states
    else:
        print(f"[{config['name']}] — consistent ✓")
        config["adjusted_states"] = states

=== CIB Consistency Check ===

[Evolutionary Optimistic] — consistent ✓
[Evolutionary Conservative] — consistent ✓
[AI-Driven Transformation] — consistent ✓
[Hardware Evolution + Regulatory Push] — consistent ✓
[Space + AI Disruption] — consistent ✓
[Fragmented Progress] — consistent ✓


In [4]:
scenarios: list[Scenario] = []

for config in scenario_configs:
    print(f"\n{'='*60}")
    print(f"Generating: {config['name']}")
    print(f"{'='*60}")

    # use CIB-adjusted states
    adjusted_states = config["adjusted_states"]

    assumptions = []
    for i, d in enumerate(config["drivers"]):
        state = adjusted_states[i]
        assumptions.append(DriverAssumption(
            driver_id=d.id,
            state=state,
            description=f"{d.name}: {state}",
        ))

    assumptions_text = "\n".join([
        f"- {a.description} (Driver origin: {next((d.origin.value for d in drivers if d.id == a.driver_id), '?')})"
        for a in assumptions
    ])

    # RAG retrieval for grounding
    driver_names = " ".join([d.name for d in config["drivers"]])
    query_emb = embed([f"{driver_names} regulatory frequency monitoring 2035"])[0]
    rag = collection.query(query_embeddings=[query_emb], n_results=4, include=["documents", "metadatas"])
    rag_text = "\n\n---\n\n".join([
        f"[Chunk ID: {rag['ids'][0][i]}] (Source: {rag['metadatas'][0][i]['source_title']})\n{rag['documents'][0][i]}"
        for i in range(len(rag["ids"][0]))
    ])
    rag_chunk_ids = rag["ids"][0]

    prompt = prompts.SCENARIO_GENERATE.format(
        driver_assumptions=assumptions_text,
        scenario_type=config["type"].value,
        rag_chunks=rag_text,
    )

    result = safe_chat_json(prompt, system="You are a technology foresight expert writing vivid future scenarios for spectrum monitoring in 2035.")

    # merge source IDs: driver sources + RAG sources + LLM-cited sources
    all_source_ids = list(rag_chunk_ids)
    for d in config["drivers"]:
        all_source_ids.extend(d.source_chunk_ids)
    all_source_ids.extend(result.get("source_chunk_ids_used", []))

    scenario = Scenario(
        title=result.get("title", config["name"]),
        narrative=result.get("narrative", ""),
        type=config["type"],
        assumptions=assumptions,
        source_chunk_ids=list(set(all_source_ids)),
    )
    scenarios.append(scenario)

    print(f"\nTitle: {scenario.title}")
    print(f"Narrative preview: {scenario.narrative[:300]}...")
    print(f"Key changes: {result.get('key_changes', [])}")
    print(f"Source chunks: {len(scenario.source_chunk_ids)}")

print(f"\n=== Generated {len(scenarios)} scenarios ===")


Generating: Evolutionary Optimistic



Title: Seamless, Ultra-Wideband Distributed Spectrum Monitoring in 2035
Narrative preview: By 2035, spectrum monitoring has evolved into a seamless, highly distributed, and ultra-wideband operation powered by breakthroughs in photonic signal processing, advanced software-defined radio (SDR) architectures, and edge computing. Regulatory authorities no longer rely on bulky, centralized moni...
Key changes: ['Deployment of distributed, photonic ultra-wideband sensor nodes with edge AI processing', 'Transition from hardware-defined radios to fully software-defined, reconfigurable receivers', 'Integration of space-based LEO satellite constellations with ground networks for global 3D spectrum awareness', 'Real-time, terahertz-scale spectrum scanning with nanosecond-level signal detection and automatic attenuation control', 'Advanced AoA error correction and TDOA geolocation enabled by GNSS and quantum timing synchronization']
Source chunks: 10

Generating: Evolutionary Conservative



Title: Seamless, Real-Time Global Spectrum Oversight in 2035
Narrative preview: By 2035, regulatory frequency monitoring has evolved into a highly integrated, real-time, and globally distributed system leveraging advances in photonic signal processing, edge computing, and AI-driven analytics. Ultra-wideband receivers now routinely employ photonic ADCs capable of sampling rates ...
Key changes: ['Deployment of photonic ADC-based ultra-wideband receivers with real-time multi-gigahertz bandwidth capture', 'Distributed edge computing nodes processing I/Q data locally, integrated with 6G infrastructure', 'Fully software-defined radios with rapid reconfiguration and AI-driven automatic signal classification', 'Enhanced direction finding using angle-of-arrival error correction and TDOA with GNSS integration', 'Obsolescence of manual scanning and centralized data processing in favor of continuous automated monitoring']
Source chunks: 10

Generating: AI-Driven Transformation



Title: AI-Driven, Error-Corrected Spectrum Monitoring in a Software-Defined Regulatory Landscape
Narrative preview: By 2035, regulatory frequency monitoring has undergone a profound transformation driven by breakthroughs in digital downconversion, software-defined radio (SDR) architectures, and advanced AI-powered dynamic range enhancement systems. The once hardware-heavy, rigid monitoring stations have evolved i...
Key changes: ['Transition to fully software-defined radio architectures with advanced digital downconversion enabling rapid reconfiguration and multi-channel processing', 'Breakthrough angle-of-arrival error correction and TDOA processing delivering meter-level geolocation accuracy', 'Energy-efficient AI architectures powering federated learning across LEO satellite constellations and ground stations for decentralized, privacy-preserving spectrum analysis', 'Obsolescence of photonic signal processing and tunable bandpass filters due to stagnation, with reliance on advanced


Title: Seamless, AI-Driven Global Spectrum Oversight in 2035
Narrative preview: By 2035, spectrum monitoring has undergone a profound transformation driven by breakthroughs in photonic signal processing, ultra-wideband real-time capture, and edge computing integrated with AI-native 6G networks. Regulatory authorities now operate within a fully distributed, intelligent monitorin...
Key changes: ['Deployment of ultra-wideband photonic receivers enabling real-time, full-spectrum capture with high dynamic range', 'Distributed edge computing with I/Q data streaming and federated AI models for local and global spectrum analysis', 'Transition from manual scanning to AI-driven anomaly detection and autonomous enforcement actions', 'Integration of space-based LEO satellite constellations with terrestrial monitoring for comprehensive 3D spectrum awareness', 'Evolution of regulatory operator roles from manual monitoring to strategic oversight via immersive AI dashboards']
Source chunks: 11

Gene


Title: AI-Native 6G Networks and Photonic Intelligence Transform Regulatory Spectrum Monitoring in 2035
Narrative preview: By 2035, regulatory frequency monitoring has undergone a radical transformation driven by the convergence of AI-native 6G intelligent networks, photonic signal processing breakthroughs, and advanced machine learning-integrated software-defined radios (SDRs). Traditional spectrum monitoring architect...
Key changes: ['Transition from traditional digital downconversion and I/Q streaming to photonic ultra-wideband signal processing with AI-native SDRs', 'Deployment of federated AI-coordinated space-air-ground networks providing global, real-time 3D spectrum awareness', 'Operators shift from manual data analysis to immersive AI-driven interfaces delivering actionable spectrum insights', 'Obsolescence of legacy SDR architectures and stagnation in real-time bandwidth capture hardware, replaced by ML-enabled adaptive radios', 'Integration of edge computing with federated


Title: Quantum-Enhanced Distributed Spectrum Guardianship in 2035
Narrative preview: By 2035, regulatory frequency monitoring has undergone a radical transformation driven by breakthroughs in photonic signal processing and energy-efficient AI architectures, while traditional SDR evolution and angle-of-arrival error correction have stagnated. Regulatory authorities now operate within...
Key changes: ['Breakthrough photonic signal processing enables ultra-wideband real-time spectrum capture beyond electronic ADC limits.', 'Distributed edge computing processes massive I/Q data streams locally, reducing bandwidth and enabling real-time decisions.', 'AI-driven dynamic spectrum access with energy-efficient federated learning replaces manual spectrum management.', 'Legacy SDR evolution and angle-of-arrival error correction stagnate, shifting focus to TDOA and GNSS-based localization.', 'Integration of LEO satellite constellations with ground networks provides comprehensive 3D global spectrum

In [5]:
# save scenarios
scenario_state = {
    "scenarios": [s.model_dump(mode="json") for s in scenarios],
}
with open("../data/outputs/scenario_state.json", "w") as f:
    json.dump(scenario_state, f, indent=2)
print(f"Saved {len(scenarios)} scenarios")

Saved 6 scenarios
